# OrganAMNIST classification with an MLP

Module 2 warm-up exercise. Baseline for the CNN notebook to beat.

- **Data:** OrganAMNIST, 28x28 grayscale, 11 organ classes (auto-downloads).
- **Model:** one hidden layer, 784 -> 512 -> 11.
- **Training:** Adam, cross-entropy, early stopping with patience 5.

Reusable code lives in `src/`; the first code cell locates the repo root automatically.

In [ ]:
import os
# moved into notebooks/; walk up to the repo root so src/ and data/ resolve regardless of launch cwd
while not os.path.exists("requirements.txt") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.data import load_medmnist
from src.metrics import multiclass_metrics, plot_confusion, plot_history
from src.models import MLP, AlexNet
from src.seed import get_device, set_seed
from src.training import evaluate, fit

set_seed(27)
device = get_device()
print("device:", device)

In [ ]:
BATCH_SIZE = 128
LR = 5e-4
N_EPOCHS = 30
HIDDEN_SIZES = (512,)
PATIENCE = 5

train_loader, val_loader, test_loader, info = load_medmnist(
    "organamnist", batch_size=BATCH_SIZE, image_size=28
)
n_classes = len(info["label"])
n_channels = info["n_channels"]

print(f"task: {info['task']}  classes: {n_classes}  channels: {n_channels}")
print(f"train {len(train_loader.dataset)} | val {len(val_loader.dataset)} | test {len(test_loader.dataset)}")

In [ ]:
import random

images, labels = next(iter(train_loader))
class_names = [info["label"][str(i)] for i in range(n_classes)]

fig, axes = plt.subplots(4, 4, figsize=(7, 7))
for ax, idx in zip(axes.ravel(), random.sample(range(len(images)), 16)):
    ax.imshow(images[idx].squeeze() * 0.5 + 0.5, cmap="gray")
    ax.set_title(class_names[labels[idx].item()], fontsize=7)
    ax.axis("off")
fig.tight_layout()

In [ ]:
input_size = n_channels * 28 * 28

model = MLP(input_size, n_classes, hidden_sizes=HIDDEN_SIZES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## Train

`fit` reports the **epoch-average** loss (not the last batch), stops when validation
loss has not improved for 5 epochs, and restores the best weights before returning.

In [ ]:
history = fit(model, train_loader, val_loader, criterion, optimizer, device,
              n_epochs=N_EPOCHS, patience=PATIENCE)

plot_history(history)

## Test

Only run this once, at the end. Expect a large drop from validation to test accuracy:
the OrganAMNIST test split comes from **different patients** than train/val, so it
measures generalisation across patients, not just across images.

In [ ]:
test_acc, test_loss, y_true, y_prob = evaluate(model, test_loader, criterion, device)

print(f"Test accuracy: {test_acc:.2f}%   Test loss: {test_loss:.4f}")
for name, value in multiclass_metrics(y_true, y_prob).items():
    print(f"{name:>10}: {value:.4f}")

fig, ax = plt.subplots(figsize=(8, 8))
plot_confusion(y_true, y_prob.argmax(1), class_names=class_names, ax=ax, normalize="true")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()

## Things to try (from the slides)

- Swap the dataset: `breastmnist` (binary), `bloodmnist`, `dermamnist`, `retinamnist`.
- More hidden layers: `HIDDEN_SIZES = (512, 256)`. Does deeper help here?
- Add `dropout=0.3` to the `MLP(...)` call and compare the train/val gap.
- Note the parameter count above, then compare it with AlexNet in the next notebook.